# 第78课 · 教 AI 跟着音乐点头——onset 包络与自相关（autocorrelation），从零估出 BPM

**学习目标**
1. 理解 onset（音符起点）检测的原理：谱通量（Spectral Flux）
2. 实现 onset_envelope（纯 NumPy + aurora.audio.stft）
3. 用自相关估计节拍周期和 BPM
4. 与 aurora.music.beat_track 对比验证

> onset 曲线人话：「能量突然抬起」；rubato/变速会失败——记下边界。

← **上一课**　[L77 · 音乐特征工程](L77_music_features.ipynb)

> 上节课学习了 **音乐特征工程**：chroma、RMS 能量、ZCR，调用 aurora.music 从零实现。  
> 本课将探讨 **节拍追踪从零实现**。

## 本课剧情：AI 怎么抓住音乐里重复出现的“重拍”？

你打快板，别人能跟着点头——这是因为音乐有规律的**能量爆发**：鼓击、弹拨、呼吸音。大脑在找这些"能量突增时刻"，物理上叫 **Onset**。

**核心思路：两步走**

```
音频信号
   ↓ STFT → 幅度谱（每帧）
   ↓ 谱通量（Spectral Flux）= 相邻帧幅度谱之差的正值之和
   ↓ onset_envelope（每帧一个值，大=有强拍，小=安静）
   ↓ 自相关（Autocorrelation）找周期
   ↓ 峰值 lag → BPM = 60 / (lag × hop / sr)
```

**谱通量（Spectral Flux）**：
```
SF[t] = sum(max(|X[t]| - |X[t-1]|, 0))  按频率bin求和（幅度谱差分，与 aurora.music 一致）
```
只取正差——能量突增是onset，能量衰减不是。

**自相关找周期**：
```
R[τ] = sum(envelope[t] × envelope[t+τ])  对所有t求和
```
若节拍周期为 T 帧，则 R 在 τ=T, 2T, 3T 处有峰值。找最大峰 → BPM。

**本节任务**：用纯 NumPy 实现 `my_onset_envelope()` 和 `my_beat_track()`，在 120 BPM 合成信号上验证。

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.stft import stft

In [3]:
# 生成测试信号：120 BPM 的鼓击模拟
SR = 16000
DURATION = 4.0
HOP = 256
N_FFT = 2048

t = np.linspace(0, DURATION, int(SR * DURATION), endpoint=False)

# 模拟鼓击：每 0.5 秒一个短暂脉冲
beat_period = SR // 2   # 120 BPM
signal = np.zeros(len(t))
for beat_start in range(0, len(t), beat_period):
    end = min(beat_start + SR // 50, len(t))   # 20ms 脉冲
    signal[beat_start:end] = np.random.randn(end - beat_start) * 0.8

signal = signal / (np.abs(signal).max() + 1e-8)
print(f'信号长度: {len(signal)} 采样 ({DURATION}s @ {SR}Hz)')
print(f'期望 BPM: 120，期望节拍周期: {beat_period} 采样 = {beat_period/SR*1000:.0f}ms')

信号长度: 64000 采样 (4.0s @ 16000Hz)
期望 BPM: 120，期望节拍周期: 8000 采样 = 500ms


## ✏️ 任务 1：实现 onset_envelope

**签名**：
```python
def my_onset_envelope(signal, sample_rate, n_fft=N_FFT, hop_length=HOP) -> np.ndarray:
    # 返回: (n_frames,) float，谱通量包络
```

**4步实现路线**：

| 步骤 | 操作 | 工具 |
|---|---|---|
| 1 | STFT → 幅度谱 | `np.abs(stft(signal, n_fft=n_fft, hop_length=hop_length))` |
| 2 | 相邻帧幅度谱之差 | `np.diff(mag, axis=0)` → shape (n_frames-1, n_bins) |
| 3 | 只保留正差值（能量增加） | `np.maximum(diff, 0)` |
| 4 | 每帧求和 | `.sum(axis=1)` → (n_frames-1,) |

**验收标准**：
- 鼓击信号（120 BPM）：envelope 在鼓击时刻有明显峰值
- 输出 shape = `(n_frames-1,)`（比帧数少1，因为做了diff）
- 全零信号 → envelope 全为 0

In [4]:
def my_onset_envelope(signal, sample_rate, n_fft=N_FFT, hop_length=HOP):
    """谱通量 onset 包络：相邻帧幅度谱正差值之和。"""
    # stft 返回形状 (n_frames, n_fft//2+1)；在 axis=0（时间轴）做差分，axis=1（频率轴）求和
    # ✏️ TODO：调用 stft，取绝对值，计算差分，半波整流，按频率求和
    raise NotImplementedError("TODO")

try:
    env = my_onset_envelope(signal, SR)
    fps = SR / HOP
    times = np.arange(len(env)) / fps
    print(f'onset envelope shape: {env.shape}')
    print(f'帧率: {fps:.1f} fps')
    # ✅ 验证 shape 和值域
    assert env.ndim == 1, "env 应为 1D 数组"
    assert (env >= 0).all(), "谱通量必须非负（半波整流后应 ≥ 0）"
    n_frames = stft(signal, n_fft=N_FFT, hop_length=HOP).shape[0]
    assert env.shape[0] == n_frames - 1, (
        f"env 长度应为 n_frames-1={n_frames-1}，实际为 {env.shape[0]}；"
        "原因：np.diff 在时间轴上会少一帧"
    )
    print('✅ Task 1 验证通过')
    plt.figure(figsize=(10, 2))
    plt.plot(times, env)
    plt.xlabel('时间 (s)')
    plt.title('Onset Envelope（谱通量）')
    plt.tight_layout()
    plt.show()
except (NotImplementedError, TypeError):
    print('⚠️  请先实现 my_onset_envelope，再运行此 cell。')
    env, times = None, None
    fps = SR / HOP


⚠️  请先实现 my_onset_envelope，再运行此 cell。


### 🤔 先想一想：怎么从包络里"数"出节拍？

先别看公式。假设包络每隔 T 帧就"鼓"一下，但你**并不知道 T**，怎么把它找出来？

一个笨办法：把包络**复制一份、向右平移 τ 帧**，再和原包络逐点相乘、求和——这就是自相关 `R[τ]`。

- 当 τ 恰好等于节拍周期 T：鼓点对齐鼓点 → 乘积和最大（出现峰值）。
- 当 τ 不对齐：鼓点撞到安静处 → 乘积和很小。

所以一句话直觉：**自相关 = 信号和"自己延迟版"的相似度；相似度的峰值位置 = 节拍周期。** 找到那个峰在哪，再换算成 BPM 就行了。下面的任务 2 就是把这句话写成代码。

## ✏️ 任务 2：用自相关估计 BPM

**签名**：
```python
def my_beat_track(signal, sample_rate, hop_length=HOP) -> tuple[float, np.ndarray]:
    # 返回: (bpm: float, beat_times: np.ndarray)
    # beat_times: 单位秒，每个节拍的时间位置
```

**3步实现路线**：

| 步骤 | 操作 | 关键公式 |
|---|---|---|
| 1 | 调用 `my_onset_envelope(signal, sample_rate)` 得到包络 | shape (n_frames-1,) |
| 2 | 自相关：`R[τ] = sum(envelope × envelope[τ:])` | 逐 lag 点积：`np.dot(env[:-lag], env[lag:])`（等价于 `np.correlate(e, e, 'full')` 的后半段，这里手写以看清定义） |
| 3 | 在合理 BPM 范围（40–240，与代码一致）内找最大 lag | `lag_samples = argmax(R[min_lag:max_lag])` |

**BPM 转换**：
```
lag_in_frames = argmax(R)
lag_in_seconds = lag_in_frames × hop_length / sample_rate
BPM = 60.0 / lag_in_seconds
```

**验收标准**：
- 120 BPM 合成信号 → 估计 BPM 误差 < 15 BPM（与下方检查格 `abs(bpm - 120) < 15` 一致）
- `beat_times` 单位为秒，第一个节拍位置在合理范围内

In [5]:
def my_beat_track(signal, sample_rate, hop_length=HOP):
    """从自相关估计 BPM 和节拍时间。"""
    env = my_onset_envelope(signal, sample_rate, hop_length=hop_length)
    fps = sample_rate / hop_length

    # BPM 范围 40-240 对应的 lag 范围（单位：帧）
    min_lag = max(1, int(fps * 60 / 240))
    max_lag = min(len(env) - 1, int(fps * 60 / 40))
    lags = np.arange(min_lag, max_lag + 1)

    # ✏️ TODO — 分四步实现：
    # 1. 计算自相关向量（env 与移位 env 的点积）：
    #    ac = [np.dot(env[:-lag], env[lag:]) for lag in lags]
    # 2. 找最优 lag（自相关最大处）：
    #    best_lag = lags[np.argmax(ac)]
    # 3. 换算 BPM（公式：BPM = 60 × fps / lag）：
    #    bpm = fps * 60.0 / best_lag
    # 4. 找节拍时间（秒）：先找首拍相位，再以 best_lag 为步长向后步进
    #    pos = np.argmax(env[:best_lag])              # 首拍帧索引
    #    beat_frames = np.arange(pos, len(env), best_lag)
    #    beats = beat_frames / fps                    # 转换为秒
    #    return bpm, beats
    raise NotImplementedError("TODO")

try:
    bpm, beats = my_beat_track(signal, SR)
    print(f'估计 BPM: {bpm:.1f}  (期望: 120)')
    print(f'节拍数: {len(beats)}  (参考: ~{int(DURATION * 2)})')
    assert abs(bpm - 120) < 15, f'BPM 偏差过大: {bpm:.1f}'
    print('✅ 节拍追踪验证通过')
except (NotImplementedError, TypeError):
    print('⚠️  请先实现 my_beat_track，再运行此 cell。')
    bpm, beats = None, None


⚠️  请先实现 my_beat_track，再运行此 cell。


In [6]:
# 与 aurora.music 对比
# 注意：参考实现的 onset_envelope / beat_track 都默认使用 n_fft=2048；本课与参考口径统一
from aurora.music import onset_envelope, beat_track

ref_env = onset_envelope(signal, SR, n_fft=N_FFT, hop_length=HOP)
ref_bpm, ref_beats = beat_track(signal, SR, hop_length=HOP)

if bpm is not None:
    print(f'自实现 BPM:        {bpm:.1f}')
    print(f'aurora.music BPM: {ref_bpm:.1f}')
    print(f'误差: {abs(bpm - ref_bpm):.1f} BPM')
else:
    print(f'aurora.music 参考 BPM: {ref_bpm:.1f}  (实现后再对比)')


aurora.music 参考 BPM: 121.0  (实现后再对比)


## 本课收束

| 概念 | 要记住的 |
|---|---|
| 谱通量 | 正差值之和，检测能量突增 |
| 自相关 | lag=τ 处最大值 → 周期 τ → BPM |
| aurora.music | beat_track() 是你刚实现的算法的参考实现 |
| L79 | 有了 chroma + RMS + BPM 特征，就可以构建音乐嵌入向量 |

下一课（L79）将用对比学习（contrastive learning）训练音乐嵌入：三元组损失（triplet loss）让同类曲目在向量空间中靠近、不同类拉远。

## ✏️ 白板挑战：节拍追踪手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：谱通量（Spectral Flux）公式：SF[t]=？为什么只取正差值？  
（SF[t]=Σ max(|X[t,k]|-|X[t-1,k]|, 0)，幅度谱差分；只取正差→只检测能量增加（onset），能量衰减（offset）不是节拍）

**问 2**：120 BPM 的音乐，hop=256，sr=16000，一个节拍对应多少帧？  
（节拍周期=60/120=0.5秒；0.5×16000/256=31.25≈31帧）

**问 3**：自相关 R[τ] 的物理含义是什么？为什么节拍处有峰值？  
（R[τ]=envelope与自身错位τ后的点积；若节拍周期=T，则每隔T时刻都有能量峰，错位T后峰对齐→R[T]大）

**问 4**：若 BPM 估计为 60（实际 120），最可能的原因是什么？  
（自相关在τ=2T处的峰可能比τ=T更大——倍音程问题（octave ambiguity）；检测次谐波误判为基本周期）

**问 5**：onset_envelope 全为 0 时，BPM 估计会怎样？应该如何处理？  
（R[τ]=0对所有τ；argmax返回0→lag=0→除以0或inf BPM；应检查envelope.max()>threshold，若为0则返回nan或报错）

推导完成后运行下方格验证。

In [7]:
# ✏️ 对答案格
import numpy as np

SR_TEST = 16000
HOP_TEST = 256
BPM_TRUE = 120.0

# 问1：谱通量概念验证（构造能量增加 vs 衰减的帧）
power_prev = np.array([1.0, 2.0, 3.0])
power_curr = np.array([3.0, 1.0, 5.0])
sf = np.sum(np.maximum(power_curr - power_prev, 0))
assert sf == 4.0, f"SF应=max(2,0)+max(-1,0)+max(2,0)=4，得{sf}"
print(f"Q1 ✅  谱通量: max(2,0)+max(-1,0)+max(2,0)={sf}（只取能量增加，忽略衰减）")

# 问2：120BPM帧数计算
beat_period_sec = 60.0 / BPM_TRUE
beat_period_frames = beat_period_sec * SR_TEST / HOP_TEST
expected_frames = 31.25  # 约31帧
assert abs(beat_period_frames - expected_frames) < 0.1, f"节拍帧数={beat_period_frames:.2f}，期望≈{expected_frames}"
print(f"Q2 ✅  120BPM: 节拍={beat_period_sec}s=0.5×{SR_TEST}/{HOP_TEST}={beat_period_frames:.2f}帧")

# 问3：自相关演示
np.random.seed(42)
n_frames = 120
# 模拟120BPM包络：每~31帧一个脉冲
envelope_test = np.zeros(n_frames)
for i in range(0, n_frames, int(beat_period_frames)):
    if i < n_frames:
        envelope_test[i] = 1.0
corr = np.correlate(envelope_test, envelope_test, 'full')
half = corr[n_frames-1:]  # 正lag部分
peak_lag = np.argmax(half[5:]) + 5  # 跳过lag=0（最大但无意义）
bpm_est = 60 / (peak_lag * HOP_TEST / SR_TEST)
print(f"Q3 ✅  自相关演示: peak_lag={peak_lag}帧，估计BPM={bpm_est:.1f}（理论={BPM_TRUE}）")

# 问4：倍音程问题演示
corr_at_T = half[int(beat_period_frames)]
corr_at_2T = half[int(2*beat_period_frames)]
print(f"Q4 ✅  R[T={int(beat_period_frames)}]={corr_at_T:.2f}, R[2T={int(2*beat_period_frames)}]={corr_at_2T:.2f} — 若2T峰更大则估计为60BPM（倍音程问题）")

# 问5：全零包络处理
try:
    zeros = np.zeros(SR_TEST * 2)
    env_zeros = my_onset_envelope(zeros, SR_TEST)
    assert np.all(env_zeros == 0), f"全零信号onset_envelope应全0"
    print(f"Q5 ✅  全零信号: onset_envelope全为0（应在beat_track中检查并处理）")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  全零包络→BPM无法估计；应检查envelope.max()<threshold时返回nan（待实现后验证）")

print("\n🎉 节拍追踪白板挑战通过！")

Q1 ✅  谱通量: max(2,0)+max(-1,0)+max(2,0)=4.0（只取能量增加，忽略衰减）
Q2 ✅  120BPM: 节拍=0.5s=0.5×16000/256=31.25帧
Q3 ✅  自相关演示: peak_lag=31帧，估计BPM=121.0（理论=120.0）
Q4 ✅  R[T=31]=3.00, R[2T=62]=2.00 — 若2T峰更大则估计为60BPM（倍音程问题）
Q5 ✅  全零包络→BPM无法估计；应检查envelope.max()<threshold时返回nan（待实现后验证）

🎉 节拍追踪白板挑战通过！


In [ ]:
# ✏️ 本课自评
l78_review = {
    "spectral_flux_understood":  None,  # 理解谱通量=只取正差值（能量增加）？True/False
    "autocorr_bpm_logic":        None,  # 理解自相关找周期：R[T]大→节拍周期=T帧→BPM=60/(T×hop/sr)？True/False
    "onset_envelope_impl":       None,  # my_onset_envelope()实现正确，120BPM信号有明显峰值？True/False
    "beat_track_impl":           None,  # my_beat_track()实现正确，120BPM估计误差<15BPM？True/False
    "whiteboard_passed":         None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l78_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l78_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L78 全部通关！进入 L79：音乐嵌入向量')

---

→ **下一课**　[L79 · 音乐嵌入向量](L79_music_embed.ipynb)

> 下节课将学习 **音乐嵌入向量**：对比学习：相似风格靠近，不同流派拉远。